In [1]:
!pip install asyncpraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 5.9 MB/s eta 0:00:00


In [2]:
import asyncio
import asyncpraw

# Define subreddits and keywords
subreddits = [
    "running", "trailrunning", "ultramarathon", "Fitness", "Strava",
    "Marathon", "UltraRunning"
]
keywords = ["route", "suggestion", "distance", "length", "difficulty", "from", "recommendation", "find a route"]

# Authenticate with Reddit API
reddit = asyncpraw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT,
)

In [3]:
async def find_running_route_queries(subreddit_name):
    queries = []
    subreddit = await reddit.subreddit(subreddit_name)
    async for post in subreddit.new(limit=500):  # Get the last n posts
        # Check post title for specific keywords
        if any(keyword in post.title.lower() for keyword in keywords):
            queries.append(f"Post: {post.title}")

        # Load and check comments
        if post.num_comments > 0:
            try:
                await post.comments.replace_more(limit=None)  # Load all comments
                # Check if there are any comments to process
                if post.comments:
                    for comment in post.comments.list():
                        if any(keyword in comment.body.lower() for keyword in keywords):
                            queries.append(f"Comment in '{post.title}': {comment.body}")
                else:
                    print(f"No comments for post '{post.title}'")
            except Exception as e:
                # Ignore the error and continue to the next post if comments fail to load
                print(f"Error loading comments for post '{post.title}': {e}")

    return queries

In [4]:
# Collect queries from subreddits
async def main():
    all_queries = []
    for subreddit in subreddits:
        print(f"Fetching posts from /r/{subreddit}...")
        queries = await find_running_route_queries(subreddit)
        all_queries.extend(queries)

    print("\nRunning Route Queries Found:")
    for i, query in enumerate(all_queries, start=1):
        print(f"{i}. {query}")

# Execute the main function in the existing event loop
await main()

Fetching posts from /r/running...
Error loading comments for post 'Best books about running 📖🏃🏻‍♀️': 'NoneType' object is not iterable
Error loading comments for post 'Achievements for Monday, January 13, 2025': 'NoneType' object is not iterable
Error loading comments for post 'Official Q&A for Monday, January 13, 2025': 'NoneType' object is not iterable
Error loading comments for post 'Li'l Race Report Thread': 'NoneType' object is not iterable
Error loading comments for post 'This is for the people who Run in underwear ': 'NoneType' object is not iterable
Error loading comments for post 'Achievements for Sunday, January 12, 2025': 'NoneType' object is not iterable
Error loading comments for post 'Official Q&A for Sunday, January 12, 2025': 'NoneType' object is not iterable
Error loading comments for post 'The Weekly Training Thread': 'NoneType' object is not iterable
Error loading comments for post 'Anyone else feel a deep connection with their running route?': 'NoneType' object is n